# Downstream analysis

Paper-facing downstream classification tables and figures. HEST excludes TENX143. Hibou-L is excluded entirely, and LEMON refers only to the ViT-S/8 checkpoint. The paper-facing CellBridge all/adapted representation is labeled simply as CellBridge.

## Imports

In [ ]:
import os
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)

WORKSPACE = Path(os.environ.get(
    "CELLBRIDGE_WORKSPACE",
    "/lustre/scratch126/cellgen/lotfollahi/so25",
))
RESULTS_ROOT = Path(os.environ.get(
    "CELLBRIDGE_INFERENCE_RESULTS",
    WORKSPACE / "outputs/reproducibility/inference",
))
FIGURE_DIR = Path(os.environ.get(
    "CELLBRIDGE_DOWNSTREAM_FIGURES",
    WORKSPACE / "outputs/reproducibility/figures/downstream",
))

ID_DATASETS = ["hest"]
OOD_DATASETS = ["lizard", "nucls", "pannuke", "phenocell"]
DATASET_LABELS = {
    "hest": "HEST",
    "lizard": "Lizard",
    "nucls": "NuCLS",
    "pannuke": "PanNuke",
    "phenocell": "PhenoCell",
}

BASELINE_LABELS = {
    "uni": "UNI2-h",
    "virchow2": "Virchow2",
    "lemon": "LEMON",
    "past": "PAST",
    "volta": "VOLTA",
    "spatialwhisperer": "SpatialWhisperer",
}
LEMON_VERSION = "vits8"
BASELINE_VERSIONS = {method: "all" for method in BASELINE_LABELS}
BASELINE_VERSIONS["lemon"] = LEMON_VERSION
CELLBRIDGE_METHODS = [
    "CellBridge" if size == "all" and arm == "adapted"
    else f"CellBridge {size} ({arm})"
    for size in ("5K", "10K", "50K", "all")
    for arm in ("adapted", "projected")
]
EXPECTED_METHODS = list(BASELINE_LABELS.values()) + CELLBRIDGE_METHODS

EXCLUDED_METHODS = {"hibou_l"}
EXCLUDED_HEST_SAMPLES = {"TENX143"}

METHOD_GROUPS = [
    ("Patch-level FMs", ["UNI2-h", "Virchow2"]),
    ("Cell-level FMs", ["VOLTA", "LEMON"]),
    ("Multimodal FMs", ["PAST", "SpatialWhisperer"]),
    ("Ours", CELLBRIDGE_METHODS),
]
# The combined table carries one representative per family.
FINAL_METHOD_GROUPS = [
    ("Patch-level FMs", ["UNI2-h", "Virchow2"]),
    ("Cell-level FMs", ["VOLTA", "LEMON"]),
    ("Multimodal FMs", ["PAST", "SpatialWhisperer"]),
    ("Ours", ["CellBridge"]),
]
SHADED_GROUP = "Ours"
ROW_COLOUR = "gray!15"  # \rowcolor needs \usepackage[table]{xcolor}

# (dataset, task, column label) for the combined table.
FINAL_COLUMNS = [
    ("hest", "all", "HEST"),
    ("lizard", "all", "Lizard"),
    ("nucls", "all", "NuCLS"),
    ("pannuke", "all", "PanNuke"),
    ("phenocell", "all", "PhenoCell\\textsubscript{all}"),
    ("phenocell", "immune", "PhenoCell\\textsubscript{imm}"),
]

CLASSIFICATION_METRICS = ["macro_f1", "balanced_accuracy", "macro_auroc"]
METRIC_LABELS = {
    "macro_f1": "Macro F1",
    "balanced_accuracy": "Balanced accuracy",
    "macro_auroc": "Macro AUROC",
}
PROBES = ["linear", "knn"]
TASK = "all"  # Change to "immune" for the PhenoCell immune-only task.
HEST_COMMON_SAMPLES = True  # Compare methods on their shared retained HEST sections.
SAVE_FIGURES = True

missing_tex = [name for name in ("latex", "dvipng") if shutil.which(name) is None]
if missing_tex:
    raise RuntimeError(
        "Full LaTeX rendering requires: module load texlive-20220321 "
        f"(missing {', '.join(missing_tex)})"
    )
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 120,
    "savefig.dpi": 300,
})
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT

## Load results

A run is expected at `<method>/<version>/<dataset>` for OOD and `<method>/<version>/hest/<sample>` for separately submitted HEST jobs. Baseline versions are normally `all`; LEMON ViT-B/8 uses `vitb8`; CellBridge versions are `cells_5k`, `cells_10k`, `cells_50k`, and `cells_all`.

In [ ]:
def _require_columns(table, columns, path):
    missing = set(columns) - set(table.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {sorted(missing)}")


def _normalise_id(value):
    return str(value).strip().lower().replace("-", "_")


def paper_method(method, version, arm):
    method_id = _normalise_id(method)
    arm_id = _normalise_id(arm)
    version_id = _normalise_id(version)
    if method_id == "cellbridge" or arm_id.startswith("lora_"):
        size = version_id.removeprefix("cells_")
        size = size.upper() if size != "all" else "all"
        representation = arm_id.removeprefix("lora_")
        if size == "all" and representation == "adapted":
            return "CellBridge"
        return f"CellBridge {size} ({representation})"

    label = BASELINE_LABELS.get(method_id, str(method))
    if version_id != BASELINE_VERSIONS.get(method_id, "all"):
        label = f"{label} [{version}]"
    if arm_id not in {method_id, "default"}:
        label = f"{label} ({arm})"
    return label


def iter_runs(root):
    if not root.exists():
        raise FileNotFoundError(f"Results root does not exist: {root}")
    for method_dir in sorted(path for path in root.iterdir() if path.is_dir()):
        for version_dir in sorted(path for path in method_dir.iterdir() if path.is_dir()):
            for dataset_dir in sorted(path for path in version_dir.iterdir() if path.is_dir()):
                dataset = dataset_dir.name
                if (dataset_dir / "evaluation").is_dir():
                    yield method_dir.name, version_dir.name, dataset, None, dataset_dir
                elif dataset == "hest":
                    for sample_dir in sorted(path for path in dataset_dir.iterdir() if path.is_dir()):
                        if (sample_dir / "evaluation").is_dir():
                            yield (
                                method_dir.name, version_dir.name, dataset,
                                sample_dir.name, sample_dir,
                            )


def _contextualise(table, path, method, version, dataset, sample_id, arm, task, probe):
    table = table.copy()
    context = {
        "method": method, "version": version, "dataset": dataset,
        "arm": arm, "task": task, "probe": probe,
    }
    for column, expected in context.items():
        if column in table and not (table[column].astype(str) == str(expected)).all():
            raise ValueError(f"{path}: {column} disagrees with its directory ({expected!r})")
        table[column] = expected
    if sample_id is not None:
        if "sample_id" in table and not (table["sample_id"].astype(str) == sample_id).all():
            raise ValueError(f"{path}: sample_id disagrees with its directory ({sample_id!r})")
        table["sample_id"] = sample_id
    elif "sample_id" not in table:
        table["sample_id"] = pd.NA
    table["paper_method"] = paper_method(method, version, arm)
    table["source_path"] = str(path)
    table["result_id"] = str(path.parent)
    return table


def load_results(root):
    classification = []
    uncertainty = []
    bootstrap = []
    empty_runs = []
    metric_columns = {
        "dataset", "method", "arm", "task", "probe",
        "accuracy", "balanced_accuracy", "macro_f1",
        "weighted_f1", "macro_auroc",
    }

    for method, version, dataset, sample_id, run_dir in iter_runs(root):
        metric_paths = sorted(run_dir.glob("evaluation/*/*/*/metrics.csv"))
        if not metric_paths:
            empty_runs.append(str(run_dir))
        for path in metric_paths:
            arm, task, probe, _ = path.relative_to(run_dir / "evaluation").parts
            table = pd.read_csv(path)
            _require_columns(table, metric_columns, path)
            table = _contextualise(
                table, path, method, version, dataset, sample_id, arm, task, probe
            )
            classification.append(table)

            uncertainty_path = path.with_name("uncertainty.csv")
            if uncertainty_path.exists():
                errors = pd.read_csv(uncertainty_path)
                _require_columns(errors, {"metric", "estimate", "bootstrap_se"}, uncertainty_path)
                errors = _contextualise(
                    errors, uncertainty_path, method, version, dataset,
                    sample_id, arm, task, probe,
                )
                uncertainty.append(errors)

            bootstrap_path = path.with_name("bootstrap_metrics.csv")
            if bootstrap_path.exists():
                replicates = pd.read_csv(bootstrap_path)
                _require_columns(
                    replicates, {"bootstrap", *CLASSIFICATION_METRICS}, bootstrap_path
                )
                replicates = _contextualise(
                    replicates, bootstrap_path, method, version, dataset,
                    sample_id, arm, task, probe,
                )
                bootstrap.append(replicates)

    if not classification:
        raise FileNotFoundError(f"No canonical evaluation metrics found below {root}")
    classification = pd.concat(classification, ignore_index=True)
    uncertainty = pd.concat(uncertainty, ignore_index=True) if uncertainty else pd.DataFrame()
    bootstrap = pd.concat(bootstrap, ignore_index=True) if bootstrap else pd.DataFrame()

    classification_key = [
        "dataset", "method", "version", "arm", "task", "probe", "sample_id"
    ]
    duplicates = classification.duplicated(classification_key, keep=False)
    if duplicates.any():
        columns = classification_key + ["source_path"]
        raise ValueError(
            "Duplicate classification results found:\n"
            + classification.loc[duplicates, columns].to_string(index=False)
        )
    if not bootstrap.empty:
        bootstrap_key = classification_key + ["bootstrap"]
        duplicates = bootstrap.duplicated(bootstrap_key, keep=False)
        if duplicates.any():
            columns = bootstrap_key + ["source_path"]
            raise ValueError(
                "Duplicate bootstrap replicates found:\n"
                + bootstrap.loc[duplicates, columns].to_string(index=False)
            )
    return classification, uncertainty, bootstrap, empty_runs


In [ ]:
classification, uncertainty, bootstrap, empty_runs = load_results(RESULTS_ROOT)


def drop_excluded(table):
    """Remove excluded methods, LEMON checkpoints, and HEST sections."""
    if table.empty:
        return table
    excluded = table["method"].isin(EXCLUDED_METHODS)
    excluded |= (table["method"] == "lemon") & (table["version"] != LEMON_VERSION)
    excluded |= table["sample_id"].astype(str).isin(EXCLUDED_HEST_SAMPLES)
    return table.loc[~excluded].reset_index(drop=True)


classification = drop_excluded(classification)
uncertainty = drop_excluded(uncertainty)
bootstrap = drop_excluded(bootstrap)

observed_methods = sorted(set(classification["paper_method"]))
METHOD_ORDER = EXPECTED_METHODS + [name for name in observed_methods if name not in EXPECTED_METHODS]

hest_groups = (
    classification.loc[(classification["dataset"] == "hest") & (classification["task"] == TASK)]
    .groupby(["paper_method", "probe"])["sample_id"]
    .agg(lambda values: set(values.dropna().astype(str)))
)
HEST_SAMPLES = (
    sorted(set.intersection(*hest_groups))
    if HEST_COMMON_SAMPLES and len(hest_groups)
    else None
)
if HEST_SAMPLES is not None:
    print(f"Using {len(HEST_SAMPLES)} common HEST sections: {', '.join(HEST_SAMPLES)}")

print(
    f"Loaded {len(classification):,} downstream classification rows and "
    f"{len(bootstrap):,} OOD bootstrap replicates."
)
if empty_runs:
    print(f"{len(empty_runs)} discovered run directories do not yet contain canonical metrics.")
    display(pd.DataFrame({"run_without_metrics": empty_runs}))

result_counts = (
    classification.loc[classification["task"] == TASK]
    .groupby(["paper_method", "dataset", "probe"], dropna=False)
    .size()
    .unstack(["dataset", "probe"], fill_value=0)
    .reindex(METHOD_ORDER, fill_value=0)
)
result_counts.index.name = "Method"
display(result_counts)


## Table helpers

Classification tables show HEST mean $\pm$ section SD. OOD tables show estimate $\pm$ grouped-bootstrap SE when the corresponding `uncertainty.csv` exists. AUROC has no bootstrap row and is therefore shown as a point estimate.

In [ ]:
def _format_value(value, error=np.nan, digits=3):
    if pd.isna(value):
        return "--"
    if pd.notna(error):
        return f"{value:.{digits}f} $\\pm$ {error:.{digits}f}"
    return f"{value:.{digits}f}"


def _method_dataset_grid(rows):
    rows = rows.reindex(METHOD_ORDER)
    rows.index.name = "Method"
    return rows


def _percentage(value, error=np.nan):
    r"""One cell as ``XX.X \scriptsize{$\pm$ X.XX}`` on a 0-100 scale."""
    if pd.isna(value):
        return "--"
    text = f"{100 * value:.1f}"
    if pd.notna(error):
        text = f"{text} \\scriptsize{{$\\pm$ {100 * error:.2f}}}"
    return text


def _emphasise(text, rank):
    if rank == 1:
        return f"\\textbf{{{text}}}"
    if rank == 2:
        return f"\\underline{{{text}}}"
    return text


def _column_ranks(values):
    """Rank 1 for the best value in a column, 2 for the runner-up."""
    ordered = sorted(values.dropna().unique(), reverse=True)
    lookup = {value: rank for rank, value in enumerate(ordered[:2], start=1)}
    return values.map(lambda value: lookup.get(value, 0) if pd.notna(value) else 0)


def classification_grid(columns, metric, probe):
    """Aligned value and error frames, one column per (dataset, task) pair."""
    if metric not in classification.columns:
        raise KeyError(f"Unknown classification metric: {metric}")
    values, errors = {}, {}
    for dataset, task, label in columns:
        selected = classification.loc[
            (classification["dataset"] == dataset)
            & (classification["task"] == task)
            & (classification["probe"] == probe)
        ]
        if dataset == "hest" and HEST_SAMPLES is not None:
            selected = selected.loc[
                selected["sample_id"].astype("string").isin(HEST_SAMPLES)
            ]
        error_lookup = {}
        if not uncertainty.empty:
            rows = uncertainty.loc[
                (uncertainty["dataset"] == dataset)
                & (uncertainty["task"] == task)
                & (uncertainty["probe"] == probe)
                & (uncertainty["metric"] == metric)
            ]
            error_lookup = rows.set_index("paper_method")["bootstrap_se"].to_dict()
        for method, group in selected.groupby("paper_method", sort=False):
            if dataset == "hest":
                # HEST is summarised across sections, the OOD sets by bootstrap.
                values[(method, label)] = group[metric].mean()
                errors[(method, label)] = group[metric].std(ddof=1)
                continue
            if len(group) != 1:
                raise ValueError(
                    f"Expected one result for {(method, dataset, task, probe)}, "
                    f"found {len(group)}"
                )
            values[(method, label)] = group[metric].iloc[0]
            errors[(method, label)] = error_lookup.get(method, np.nan)

    labels = [label for _, _, label in columns]

    def frame(mapping):
        if not mapping:
            return pd.DataFrame(columns=labels, dtype=float)
        return pd.Series(mapping, dtype=float).unstack().reindex(columns=labels)

    return frame(values), frame(errors)


def _group_members(groups):
    """Yield (group, result key, display label) for each listed method."""
    for group, members in groups:
        for member in members:
            key, label = member if isinstance(member, tuple) else (member, member)
            yield group, key, label


def grouped_table(values, errors, groups=None, include_other=True):
    """Format a grid into grouped rows, marking the best and runner-up.

    Ranks are computed over the rows the table actually shows, so a trimmed
    table still marks a best and a runner-up among its own entries.
    """
    groups = METHOD_GROUPS if groups is None else groups
    listed = list(_group_members(groups))
    if include_other:
        known = {key for _, key, _ in listed}
        listed += [
            ("Other", name, name) for name in values.index if name not in known
        ]

    shown = [key for _, key, _ in listed if key in values.index]
    values, errors = values.loc[shown], errors.loc[shown]
    ranks = values.apply(_column_ranks)

    rows, order, seen = {}, [], set()
    for group, key, label in listed:
        if key not in values.index:
            continue
        if group not in seen:
            header = f"\\textbf{{{group}}}"
            rows[header] = dict.fromkeys(values.columns, "")
            order.append(header)
            seen.add(group)
        row = label
        if group == SHADED_GROUP:
            row = f"\\rowcolor{{{ROW_COLOUR}}} {row}"
        rows[row] = {
            column: _emphasise(
                _percentage(values.loc[key, column], errors.loc[key, column]),
                ranks.loc[key, column],
            )
            for column in values.columns
        }
        order.append(row)
    table = pd.DataFrame.from_dict(rows, orient="index").reindex(order)
    table.index.name = None  # keeps to_latex to a single header row
    return table


def classification_table(columns, metric, probe, groups=None, include_other=True):
    """Grouped percentage table for one metric and probe."""
    return grouped_table(
        *classification_grid(columns, metric, probe),
        groups=groups,
        include_other=include_other,
    )


def emit_latex(table, caption, label, fit=True):
    """Show the table and print its LaTeX, ruled between groups and scaled."""
    display(table)
    latex = table.to_latex(
        escape=False, na_rep="--", caption=caption, label=label,
        position="t", column_format="l" + "c" * len(table.columns),
    )
    lines = []
    seen_group = False
    for line in latex.splitlines():
        # group headers are the only rows whose first cell is bold
        if line.startswith("\\textbf{"):
            if seen_group:
                lines.append("\\midrule")
            seen_group = True
        if fit and line.startswith("\\begin{tabular}"):
            lines.append("\\resizebox{\\textwidth}{!}{%")
        lines.append(line)
        if fit and line.startswith("\\end{tabular}"):
            lines.append("}")
    print("\n".join(lines))


def emit_transposed_latex(columns, metric, probe, caption, label,
                          groups=None, fit=True):
    """Print the combined table with methods across the top and benchmarks down.

    Group names become spanning headers, and the shaded family becomes a shaded
    column, since \rowcolor cannot express a column.
    """
    groups = FINAL_METHOD_GROUPS if groups is None else groups
    values, errors = classification_grid(columns, metric, probe)
    listed = [
        (group, key, name)
        for group, key, name in _group_members(groups)
        if key in values.index
    ]
    values = values.loc[[key for _, key, _ in listed]]
    errors = errors.loc[[key for _, key, _ in listed]]
    ranks = values.apply(_column_ranks)

    spans = []
    for group, _, _ in listed:
        if spans and spans[-1][0] == group:
            spans[-1][1] += 1
        else:
            spans.append([group, 1])

    body = pd.DataFrame(
        {
            name: [
                _emphasise(
                    _percentage(values.loc[key, column], errors.loc[key, column]),
                    ranks.loc[key, column],
                )
                for column in values.columns
            ]
            for _, key, name in listed
        },
        index=list(values.columns),
    )
    display(body)

    shaded = {name for group, _, name in listed if group == SHADED_GROUP}
    spec = "l" + "".join(
        f">{{\\columncolor{{{ROW_COLOUR}}}}}c" if name in shaded else "c"
        for name in body.columns
    )
    lines = ["\\begin{table}[t]", f"\\caption{{{caption}}}", f"\\label{{{label}}}"]
    if fit:
        lines.append("\\resizebox{\\textwidth}{!}{%")
    lines += [f"\\begin{{tabular}}{{{spec}}}", "\\toprule"]
    # a single-column group stays a plain cell; \multicolumn would drop its shading
    lines.append(" & " + " & ".join(
        f"\\multicolumn{{{span}}}{{c}}{{\\textbf{{{group}}}}}" if span > 1
        else f"\\textbf{{{group}}}"
        for group, span in spans
    ) + " \\\\")
    rules, start = [], 2
    for _, span in spans:
        rules.append(f"\\cmidrule(lr){{{start}-{start + span - 1}}}")
        start += span
    lines.append(" ".join(rules))
    lines.append(" & " + " & ".join(body.columns) + " \\\\")
    lines.append("\\midrule")
    for benchmark, row in body.iterrows():
        lines.append(f"{benchmark} & " + " & ".join(row) + " \\\\")
    lines += ["\\bottomrule", "\\end{tabular}"]
    if fit:
        lines.append("}")
    lines.append("\\end{table}")
    print("\n".join(lines))


## ID classification tables (HEST)

In [ ]:
ID_COLUMNS = [(dataset, TASK, DATASET_LABELS[dataset]) for dataset in ID_DATASETS]

for probe in PROBES:
    for metric in CLASSIFICATION_METRICS:
        table = classification_table(ID_COLUMNS, metric, probe)
        emit_latex(
            table,
            caption=(
                f"ID {METRIC_LABELS[metric]} using the {probe} probe."
            ),
            label=f"tab:id-{metric.replace('_', '-')}-{probe}",
        )


## OOD classification tables

In [ ]:
OOD_COLUMNS = [(dataset, TASK, DATASET_LABELS[dataset]) for dataset in OOD_DATASETS]

for probe in PROBES:
    for metric in CLASSIFICATION_METRICS:
        table = classification_table(OOD_COLUMNS, metric, probe)
        emit_latex(
            table,
            caption=(
                f"OOD {METRIC_LABELS[metric]} using the {probe} probe."
            ),
            label=f"tab:ood-{metric.replace('_', '-')}-{probe}",
        )


## Combined tables (ID and OOD)


In [ ]:
for probe in PROBES:
    for metric in CLASSIFICATION_METRICS:
        emit_transposed_latex(
            FINAL_COLUMNS, metric, probe,
            caption=(
                f"{METRIC_LABELS[metric]} across every benchmark "
                f"using the {probe} probe."
            ),
            label=f"tab:all-{metric.replace('_', '-')}-{probe}",
        )


## Downstream evaluation distributions

Each dataset/task and metric receives one boxplot. Models are shown on the x-axis and use the fixed paper palette. Linear and $k$NN probes are grouped within each model; solid boxes denote linear and hatched boxes denote $k$NN. The diamond marks the reported point estimate (the HEST section mean or the fixed-split OOD estimate).

The distributions have different, explicit interpretations: HEST boxes contain the 14 retained section-level observations and therefore show cross-section heterogeneity; OOD boxes contain 1,000 stored grouped-bootstrap replicates and therefore show sampling uncertainty for the fixed test split. No pseudo-replicates are constructed. All text is rendered by LaTeX.

In [ ]:
PLOT_METHODS = [
    "UNI2-h",
    "Virchow2",
    "VOLTA",
    "LEMON",
    "PAST",
    "SpatialWhisperer",
    "CellBridge",
]
METHOD_COLORS = {
    "UNI2-h": "#CB722F",
    "Virchow2": "#93300C",
    "VOLTA": "#8859AB",
    "LEMON": "#D6A238",
    "PAST": "#E5A3B3",
    "SpatialWhisperer": "#287DC5",
    "CellBridge": "#88C2AB",
}
FIGURE_PROBES = ["linear", "knn"]
PROBE_LABELS = {"linear": "Linear", "knn": r"$k$NN"}
PROBE_HATCHES = {"linear": "", "knn": "///"}
METRIC_AXIS_LABELS = {
    "macro_f1": r"Macro $F_1$ (\%)",
    "balanced_accuracy": r"Balanced accuracy (\%)",
    "macro_auroc": r"Macro AUROC (\%)",
}
PLOT_COLUMNS = [
    ("hest", "all", "HEST", "hest"),
    ("lizard", "all", "Lizard", "lizard"),
    ("nucls", "all", "NuCLS", "nucls"),
    ("pannuke", "all", "PanNuke", "pannuke"),
    ("phenocell", "all", "PhenoCell (all)", "phenocell_all"),
    ("phenocell", "immune", "PhenoCell (immune)", "phenocell_immune"),
]


def metric_distributions(dataset, task, metric, probe):
    table = classification if dataset == "hest" else bootstrap
    selected = table.loc[
        (table["dataset"] == dataset)
        & (table["task"] == task)
        & (table["probe"] == probe)
    ]
    if dataset == "hest" and HEST_SAMPLES is not None:
        selected = selected.loc[
            selected["sample_id"].astype("string").isin(HEST_SAMPLES)
        ]
    return {
        method: group[metric].dropna().to_numpy(float)
        for method, group in selected.groupby("paper_method", sort=False)
    }


def plot_downstream_metric(dataset, task, dataset_label, slug, metric):
    column_key = (dataset, task, dataset_label)
    x = np.arange(len(PLOT_METHODS), dtype=float)
    offsets = np.linspace(-0.18, 0.18, len(FIGURE_PROBES))
    width = 0.28
    all_values = []
    fig, ax = plt.subplots(figsize=(9.8, 5.5))

    for probe_index, probe in enumerate(FIGURE_PROBES):
        distributions = metric_distributions(dataset, task, metric, probe)
        missing = [method for method in PLOT_METHODS if method not in distributions]
        if missing:
            raise ValueError(
                f"Missing {probe} {metric} distributions for {dataset_label}: {missing}"
            )
        data = [100 * distributions[method] for method in PLOT_METHODS]
        expected_count = 14 if dataset == "hest" else 1000
        wrong_counts = {
            method: len(values)
            for method, values in zip(PLOT_METHODS, data)
            if len(values) != expected_count
        }
        if wrong_counts:
            raise ValueError(
                f"Unexpected {probe} distribution sizes for {dataset_label}: {wrong_counts}"
            )
        positions = x + offsets[probe_index]
        boxes = ax.boxplot(
            data,
            positions=positions,
            widths=width,
            patch_artist=True,
            manage_ticks=False,
            showfliers=False,
            medianprops={"color": "#111111", "linewidth": 1.2},
            whiskerprops={"color": "#333333", "linewidth": 0.8},
            capprops={"color": "#333333", "linewidth": 0.8},
            boxprops={"edgecolor": "#222222", "linewidth": 0.75},
            zorder=2,
        )
        for box, method in zip(boxes["boxes"], PLOT_METHODS):
            box.set_facecolor(METHOD_COLORS[method])
            box.set_hatch(PROBE_HATCHES[probe])
            box.set_alpha(0.92)

        if dataset == "hest":
            rng = np.random.default_rng(100 + probe_index)
            for position, method, values in zip(positions, PLOT_METHODS, data):
                jitter = rng.uniform(-0.035, 0.035, size=len(values))
                ax.scatter(
                    np.full(len(values), position) + jitter,
                    values,
                    s=9,
                    color=METHOD_COLORS[method],
                    edgecolor="#222222",
                    linewidth=0.25,
                    alpha=0.45,
                    zorder=3,
                )

        point_values, _ = classification_grid([column_key], metric, probe)
        estimates = 100 * point_values[dataset_label].reindex(PLOT_METHODS)
        if estimates.isna().any():
            raise ValueError(
                f"Missing {probe} {metric} point estimates for {dataset_label}"
            )
        ax.scatter(
            positions,
            estimates.to_numpy(),
            marker="D",
            s=25,
            facecolor="white",
            edgecolor="#111111",
            linewidth=0.8,
            zorder=4,
        )
        all_values.extend(data)
        all_values.append(estimates.to_numpy())

    finite = np.concatenate(all_values)
    finite = finite[np.isfinite(finite)]
    lower, upper = float(finite.min()), float(finite.max())
    padding = max(1.0, 0.08 * (upper - lower))

    ax.set_xticks(x, PLOT_METHODS, rotation=20, ha="right")
    ax.set_ylabel(METRIC_AXIS_LABELS[metric])
    ax.set_title(f"{dataset_label}: {METRIC_LABELS[metric]}")
    ax.set_ylim(max(0, lower - padding), min(100, upper + padding))
    ax.grid(axis="y", alpha=0.20, linewidth=0.7)
    ax.set_axisbelow(True)
    handles = [
        Patch(
            facecolor="#B8B8B8",
            edgecolor="#222222",
            hatch=PROBE_HATCHES[probe],
            label=PROBE_LABELS[probe],
        )
        for probe in FIGURE_PROBES
    ]
    handles.append(
        Line2D(
            [], [], marker="D", linestyle="", markersize=5,
            markerfacecolor="white", markeredgecolor="#111111",
            label="Reported estimate",
        )
    )
    ax.legend(
        handles=handles,
        title="Probe",
        frameon=False,
        loc="center left",
        bbox_to_anchor=(1.01, 0.5),
        borderaxespad=0,
    )
    fig.tight_layout(rect=(0, 0, 0.83, 1))
    if SAVE_FIGURES:
        for suffix in ("png", "pdf"):
            fig.savefig(
                FIGURE_DIR / f"downstream_{slug}_{metric}.{suffix}",
                bbox_inches="tight",
            )
    plt.show()


for dataset, task, dataset_label, slug in PLOT_COLUMNS:
    for metric in CLASSIFICATION_METRICS:
        plot_downstream_metric(dataset, task, dataset_label, slug, metric)


## Raw rows

Use these frames for follow-up analyses without reloading files. Source paths are retained for provenance.

In [ ]:
display(classification.head())
display(uncertainty.head())
display(bootstrap.head())